<a id="topo"></a>

# Otimização Inteligente do PNAE — Exploração e Preparação de Dados do Censo Escolar

**Aluna:** Daniela Barreto Araújo

**Orientador:** Renato Cividini Matthiesen

**Curso:** MBA em Data Science & Analytics para Operações

**Projeto:** Otimização inteligente do PNAE: integração de modelos preditivos e programação linear na alimentação escolar

Este notebook documenta a etapa de **ingestão, tratamento (ETL) e análise exploratória** da base do **Censo Escolar**, servindo de insumo (dados de demanda: escolas, matrículas e regime de ensino) para a etapa seguinte de modelagem por Programação Linear (Problema da Dieta).

> Este notebook é o segundo módulo de dados do projeto — o primeiro, referente aos **dados de alimentos e composição nutricional**, é tratado em notebook separado. Ambos alimentam o mesmo modelo de otimização: este aqui define **quem** precisa ser alimentado, **quantas refeições** e qual o valor do repasse; o de alimentos define **com o quê** e **a que custo**.

Consulte o **[Índice](#indice)** logo abaixo para navegar diretamente até cada etapa.


<a id="indice"></a>
## Índice

1. [Ferramentas e caso de referência](#ferramentas)
2. [Importação de bibliotecas e configurações iniciais](#bibliotecas)
3. [Roteiro de Ingestão e ETL](#roteiro-etl)
4. [Perguntas de pesquisa endereçadas neste notebook](#perguntas)
5. [Carregar e consolidar os microdados do Censo Escolar](#carregar-dados)
6. [Limpeza de linhas — recorte geográfico e administrativo](#linhas)
7. [Padronização e seleção de colunas](#colunas)
8. [Dados faltantes e inconsistências](#dados-faltantes)
9. [Regras de repasse financeiro do PNAE (Resolução CD/FNDE nº 1/2026)](#regras-repasse)
10. [Cálculo do repasse financeiro por escola](#calculo-repasse)
11. [Demanda nutricional por escola — grade para a Programação Linear](#demanda-nutricional)
12. [Resumo municipal e validação cruzada](#resumo-municipal)
13. [Observações, limitações conhecidas e pontos em aberto para o TCC](#observacoes)

### Arquivos de entrada (leitura — não alterar)
Devem existir em `./data/input/` antes da execução:

| Arquivo | Conteúdo |
| --- | --- |
| `Tabela_Escola_2025.csv` | Cadastro de escolas (tabela âncora do join) |
| `Tabela_Turma_2025.csv` | Turmas |
| `Tabela_Matricula_2025.csv` | Matrículas |
| `Tabela_Gestor_Escolar_2025.csv` | Gestores escolares |
| `Tabela_Docente_2025.csv` | Docentes |

### Arquivos gerados (saída — schema mantido)
Gravados em `./data/output/` na ordem em que aparecem no notebook:

| Arquivo | Gerado em | Conteúdo |
| --- | --- | --- |
| `microdados_ed_basica_2025_original.csv` | [Carregar e consolidar](#carregar-dados) | Cache bruto da consolidação das 5 tabelas (funciona como *checkpoint*: se existir, a consolidação não é refeita) |
| `dataset_censo_salvador.csv` | [Padronização e seleção de colunas](#colunas) | Base filtrada (Salvador/BA, rede municipal, em atividade) e com colunas padronizadas |
| `resumo_repasse_pnae_amostra.csv` | [Cálculo do repasse financeiro](#calculo-repasse) | Repasse financeiro estimado por escola, por modalidade |
| `demanda_nutricional_escolas.csv` | [Demanda nutricional](#demanda-nutricional) | Demanda nutricional por escola × grupo × regime — **insumo direto (RHS) da Programação Linear** |


<a id="ferramentas"></a>
## Ferramentas
Como ferramental, estou utilizando Jupyter Notebook rodando através do Visual Studio. <br>

## Caso de referência
Para o desenvolvimento do projeto utilizaremos os dados relativos ao PNAE. <br>

Elaborar o processo exploratório de dados no papel de um cientista. <br>

https://download.inep.gov.br/dados_abertos/microdados_censo_escolar_2025_.zip <br>


# Import

Começaremos importando as bibliotecas que utilizaremos neste projeto. <br>
Estão omitidos os comandos de instalação. Portanto, ao replicar o código, lembrar de instalar (pip install) antes de importar. <br>


<a id="roteiro-etl"></a>
## Roteiro de Ingestão e ETL

**Ingestão**
- Automatize o download dos arquivos (o que for possível)
- Defina o que interessa dentro dos arquivos (aba da planilha)

**ETL**
- Descarte colunas desnecessárias ou organize tabelas de relacionamento
- Faça o tratamento adequado dos formatos de dados (float)
- Verifique dados faltantes, defina estratégia (descarte ou imputação) e métodos para preenchimento (enriquecimento com outra fonte, estimativa, etc.)
- Descarte linhas em branco e duplicatas

Este roteiro é o guia de leitura das seções de código a seguir: cada bullet acima corresponde a um bloco específico do notebook (ver [Índice](#indice)).


<a id="perguntas"></a>
## Perguntas de pesquisa endereçadas neste notebook

Este notebook trata do lado da **demanda** do problema (quem come, quantas refeições, que grupo nutricional) e do **repasse financeiro** disponível — os dois insumos, ao lado da base de alimentos/preços, que alimentam o modelo de Programação Linear. Das perguntas de pesquisa do projeto, este notebook contribui diretamente para:

- *Como criar um modelo matemático para o problema de elaboração de cardápios escolares considerando múltiplas restrições (econômicas, nutricionais e regulatórias)?* → aqui é construída a estrutura de **restrições de demanda** (`demanda_nutricional_escolas.csv`): quantos alunos, em qual grupo etário/nutricional e regime (parcial/integral), por escola.
- *Como garantir na solução o cumprimento da exigência da agricultura familiar?* e *Qual o ganho real, financeiro e nutricional...?* → dependem do **orçamento disponível por escola** (`resumo_repasse_pnae_amostra.csv`), calculado aqui a partir das regras de repasse da Resolução CD/FNDE nº 1/2026.
- *A solução é viável para uso por gestores e nutricionistas?* → a automatização do ETL (hoje manual em planilhas) é o primeiro passo para reduzir o tempo operacional citado no projeto de pesquisa.

Perguntas específicas que os dados deste notebook permitem responder:
- Quantos alunos, por escola, se enquadram em cada grupo nutricional e regime de ensino (comum vs. integral)?
- Qual o repasse financeiro anual estimado por escola e por modalidade de atendimento?
- Qual a divergência entre o total de alunos considerado para fins de repasse financeiro e o total considerado para fins de demanda nutricional (e por quê ela é esperada)?


<a id="bibliotecas"></a>
## Importar bibliotecas e configurações iniciais


In [1]:
# ------------------------------------------------------------
# Bibliotecas utilizadas neste notebook e para que servem:
#   pandas / numpy      -> manipulação de dados tabulares e cálculo
#   requests / BytesIO / zipfile -> (reservado) download e leitura de ZIP
#   os                   -> manipulação de pastas e caminhos
#   certifi              -> certificados SSL (uso junto de requests)
#   matplotlib.pyplot    -> gráficos exploratórios
#   unicodedata / re     -> normalização de texto (remoção de acentos)
#   openpyxl             -> leitura de planilhas Excel (Anexos do PNAE)
#   warnings             -> silenciar avisos não relevantes do openpyxl
# ------------------------------------------------------------
# importação das bibliotecas
import pandas as pd
import numpy as np

import requests
from io import BytesIO
import os
import certifi
import zipfile

import matplotlib.pyplot as plt


# Para lidar com caracteres acentuados
import unicodedata 

# Para normalização semântica e matching
import re


from openpyxl import load_workbook
import warnings

# Silenciar apenas avisos específicos do openpyxl
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")


# Configurações visuais
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

plt.rcParams["figure.figsize"] = (10, 6)


<a id="carregar-dados"></a>
## Carregar planilha de microdados do censo escolar


In [2]:
# ------------------------------------------------------------
# ENTRADA:  ./data/input/Tabela_{Escola,Turma,Matricula,Gestor_Escolar,Docente}_2025.csv
# SAÍDA:    ./data/output/microdados_ed_basica_2025_original.csv (cache)
# Esta célula NÃO deve ser alterada quanto ao conteúdo consolidado:
# qualquer mudança na lista `tabelas` ou nas chaves de join altera o
# schema do CSV gerado.
# ------------------------------------------------------------
# ============================================================
# CONSOLIDAÇÃO DOS MICRODADOS DO CENSO ESCOLAR 2025
# Fonte: 5 tabelas separadas (Escola, Turma, Matrícula, Gestor, Docente)
# Chave de join: CO_ENTIDADE + NU_ANO_CENSO
# Cache: se o arquivo consolidado já existir em output, carrega diretamente.
# ============================================================

# Padrão de pastas para projetos de dados
input_folder  = './data/input/'
output_folder = './data/output/'

# Garantir que as pastas existam
os.makedirs(input_folder,  exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

# Caminho do arquivo consolidado (cache)
consolidated_path = output_folder + 'microdados_ed_basica_2025_original.csv'

# ----------------------------------------------------------
# FLUXO DE DECISÃO: Cache vs. Consolidação
# ----------------------------------------------------------
if os.path.exists(consolidated_path):

    print(f"Arquivo consolidado encontrado. Carregando cache de: {consolidated_path}")

    df_censo = pd.read_csv(
        consolidated_path,
        sep=';',
        encoding='utf-8-sig',
        low_memory=False
    )

    print(f"Cache carregado com sucesso. Shape: {df_censo.shape}")

else:

    print("Arquivo consolidado NÃO encontrado. Iniciando consolidação das tabelas...")

    # ----------------------------------------------------------
    # DEFINIÇÃO DAS TABELAS E SEUS SEPARADORES
    # Nota: Tabela_Docente_2025 usa separador ';' (mesmo que as demais),
    # apesar de o pandas ter detectado '|' ao tentar sep='|' — confirmado
    # via inspeção raw do arquivo.
    # ----------------------------------------------------------
    tabelas = [
        {
            'nome'    : 'Escola',
            'arquivo' : input_folder + 'Tabela_Escola_2025.csv',
            'sep'     : ';',
            'papel'   : 'principal'   # Tabela âncora do join
        },
        {
            'nome'    : 'Turma',
            'arquivo' : input_folder + 'Tabela_Turma_2025.csv',
            'sep'     : ';',
            'papel'   : 'secundaria'
        },
        {
            'nome'    : 'Matricula',
            'arquivo' : input_folder + 'Tabela_Matricula_2025.csv',
            'sep'     : ';',
            'papel'   : 'secundaria'
        },
        {
            'nome'    : 'Gestor_Escolar',
            'arquivo' : input_folder + 'Tabela_Gestor_Escolar_2025.csv',
            'sep'     : ';',
            'papel'   : 'secundaria'
        },
        {
            'nome'    : 'Docente',
            'arquivo' : input_folder + 'Tabela_Docente_2025.csv',
            'sep'     : ';',
            'papel'   : 'secundaria'
        },
    ]

    # Chave de join comum entre todas as tabelas
    JOIN_KEYS = ['CO_ENTIDADE', 'NU_ANO_CENSO']

    # ----------------------------------------------------------
    # LEITURA DAS TABELAS
    # ----------------------------------------------------------
    dfs = {}
    for t in tabelas:
        print(f"  Lendo tabela '{t['nome']}' de: {t['arquivo']}")

        if not os.path.exists(t['arquivo']):
            raise FileNotFoundError(
                f"Arquivo não encontrado: {t['arquivo']}\n"
                f"Verifique se o arquivo está em '{input_folder}'."
            )

        df_temp = pd.read_csv(
            t['arquivo'],
            sep=t['sep'],
            encoding='latin1',
            low_memory=False
        )

        print(f"    Shape: {df_temp.shape}")
        dfs[t['nome']] = df_temp

    # ----------------------------------------------------------
    # CONSOLIDAÇÃO: LEFT JOIN com Escola como âncora
    # Estratégia left join garante que nenhuma escola seja
    # descartada por falta de registros nas tabelas secundárias.
    # Colunas duplicadas (ex.: NU_ANO_CENSO de múltiplas tabelas)
    # são eliminadas automaticamente — apenas as da âncora ficam.
    # ----------------------------------------------------------
    print("\nConsolidando tabelas via LEFT JOIN em CO_ENTIDADE + NU_ANO_CENSO...")

    df_censo = dfs['Escola'].copy()

    for t in tabelas:
        if t['papel'] == 'secundaria':
            nome = t['nome']
            df_sec = dfs[nome]

            # Identificar colunas novas (excluindo as chaves)
            # Evita duplicação de colunas que já existem na âncora
            cols_novas = [
                c for c in df_sec.columns
                if c not in df_censo.columns or c in JOIN_KEYS
            ]

            df_censo = df_censo.merge(
                df_sec[cols_novas],
                on=JOIN_KEYS,
                how='left',
                suffixes=('', f'_{nome}')
            )

            print(f"  + {nome} adicionado. Shape acumulado: {df_censo.shape}")

    print(f"\nConsolidação concluída. Shape final: {df_censo.shape}")

    # ----------------------------------------------------------
    # GRAVAÇÃO DO ARQUIVO CONSOLIDADO (cache para próximas execuções)
    # Gravado com sep=';' e encoding='utf-8-sig' para compatibilidade
    # com Excel e com o bloco de leitura do cache acima.
    # ----------------------------------------------------------
    df_censo.to_csv(
        consolidated_path,
        index=False,
        sep=';',
        encoding='utf-8-sig'
    )

    print(f"\nArquivo consolidado gravado em: {consolidated_path}")

    # Nota: o cache acima impede reprocessar as 5 tabelas a cada execução.
    # Para forçar uma nova consolidação (ex.: dados de entrada atualizados),
    # apague manualmente o arquivo em `consolidated_path`.

# ----------------------------------------------------------
# VISUALIZAÇÃO INICIAL (independente da origem dos dados)
# ----------------------------------------------------------
print(f"\nShape do DataFrame consolidado: {df_censo.shape}")
display(df_censo.head())


Arquivo consolidado encontrado. Carregando cache de: ./data/output/microdados_ed_basica_2025_original.csv
Cache carregado com sucesso. Shape: (145006, 942)

Shape do DataFrame consolidado: (145006, 942)


,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,NO_REGIAO_GEOG_IMED,CO_REGIAO_GEOG_IMED,NO_MESORREGIAO,CO_MESORREGIAO,NO_MICRORREGIAO,CO_MICRORREGIAO,NO_DISTRITO,CO_DISTRITO,NO_REGIAO_ADMINISTRATIVA,CO_REGIAO_ADMINISTRATIVA,NO_ENTIDADE,CO_ENTIDADE,TP_DEPENDENCIA,TP_CATEGORIA_ESCOLA_PRIVADA,TP_LOCALIZACAO,TP_LOCALIZACAO_DIFERENCIADA,DS_ENDERECO,NU_ENDERECO,DS_COMPLEMENTO,NO_BAIRRO,CO_CEP,NU_DDD,NU_TELEFONE,LATITUDE,LONGITUDE,TP_SITUACAO_FUNCIONAMENTO,CO_ORGAO_REGIONAL,DT_ANO_LETIVO_INICIO,DT_ANO_LETIVO_TERMINO,IN_VINCULO_SECRETARIA_EDUCACAO,IN_VINCULO_SEGURANCA_PUBLICA,IN_VINCULO_SECRETARIA_SAUDE,IN_VINCULO_OUTRO_ORGAO,IN_PODER_PUBLICO_PARCERIA,TP_PODER_PUBLICO_PARCERIA,IN_FORMA_CONT_TERMO_COLABORA,IN_FORMA_CONT_TERMO_FOMENTO,IN_FORMA_CONT_ACORDO_COOP,IN_FORMA_CONT_PRESTACAO_SERV,IN_FORMA_CONT_COOP_TEC_FIN,IN_FORMA_CONT_CONSORCIO_PUB,IN_FORMA_CONT_MU_TERMO_COLAB,IN_FORMA_CONT_MU_TERMO_FOMENTO,IN_FORMA_CONT_MU_ACORDO_COOP,IN_FORMA_CONT_MU_PREST_SERV,IN_FORMA_CONT_MU_COOP_TEC_FIN,IN_FORMA_CONT_MU_CONSORCIO_PUB,IN_FORMA_CONT_ES_TERMO_COLAB,IN_FORMA_CONT_ES_TERMO_FOMENTO,IN_FORMA_CONT_ES_ACORDO_COOP,IN_FORMA_CONT_ES_PREST_SERV,IN_FORMA_CONT_ES_COOP_TEC_FIN,IN_FORMA_CONT_ES_CONSORCIO_PUB,IN_MANT_ESCOLA_PRIVADA_EMP,IN_MANT_ESCOLA_PRIVADA_ONG,IN_MANT_ESCOLA_PRIVADA_OSCIP,IN_MANT_ESCOLA_PRIV_ONG_OSCIP,IN_MANT_ESCOLA_PRIVADA_SIND,IN_MANT_ESCOLA_PRIVADA_SIST_S,IN_MANT_ESCOLA_PRIVADA_S_FINS,NU_CNPJ_ESCOLA_PRIVADA,NU_CNPJ_MANTENEDORA,TP_REGULAMENTACAO,TP_RESPONSAVEL_REGULAMENTACAO,CO_ESCOLA_SEDE_VINCULADA,CO_IES_OFERTANTE,IN_LOCAL_FUNC_PREDIO_ESCOLAR,TP_OCUPACAO_PREDIO_ESCOLAR,IN_LOCAL_FUNC_SOCIOEDUCATIVO,IN_LOCAL_FUNC_UNID_PRISIONAL,IN_LOCAL_FUNC_PRISIONAL_SOCIO,IN_LOCAL_FUNC_GALPAO,TP_OCUPACAO_GALPAO,IN_LOCAL_FUNC_SALAS_OUTRA_ESC,IN_LOCAL_FUNC_OUTROS,IN_PREDIO_COMPARTILHADO,IN_AGUA_POTAVEL,IN_AGUA_REDE_PUBLICA,IN_AGUA_POCO_ARTESIANO,IN_AGUA_CACIMBA,IN_AGUA_FONTE_RIO,IN_AGUA_INEXISTENTE,IN_AGUA_CARRO_PIPA,IN_ENERGIA_REDE_PUBLICA,IN_ENERGIA_GERADOR_FOSSIL,IN_ENERGIA_RENOVAVEL,IN_ENERGIA_INEXISTENTE,IN_ESGOTO_REDE_PUBLICA,IN_ESGOTO_FOSSA_SEPTICA,IN_ESGOTO_FOSSA_COMUM,IN_ESGOTO_FOSSA,IN_ESGOTO_INEXISTENTE,IN_LIXO_SERVICO_COLETA,IN_LIXO_QUEIMA,IN_LIXO_ENTERRA,IN_LIXO_DESTINO_FINAL_PUBLICO,IN_LIXO_DESCARTA_OUTRA_AREA,IN_TRATAMENTO_LIXO_SEPARACAO,IN_TRATAMENTO_LIXO_REUTILIZA,IN_TRATAMENTO_LIXO_RECICLAGEM,IN_TRATAMENTO_LIXO_INEXISTENTE,IN_ALMOXARIFADO,IN_AREA_VERDE,IN_AREA_PLANTIO,IN_AUDITORIO,IN_BANHEIRO,IN_BANHEIRO_EI,IN_BANHEIRO_PNE,IN_BANHEIRO_FUNCIONARIOS,IN_BANHEIRO_CHUVEIRO,IN_BIBLIOTECA,IN_BIBLIOTECA_SALA_LEITURA,IN_COZINHA,IN_DESPENSA,IN_DORMITORIO_ALUNO,IN_DORMITORIO_PROFESSOR,IN_LABORATORIO_CIENCIAS,IN_LABORATORIO_INFORMATICA,IN_LABORATORIO_EDUC_PROF,IN_PATIO_COBERTO,IN_PATIO_DESCOBERTO,IN_PARQUE_INFANTIL,IN_PISCINA,IN_QUADRA_ESPORTES,IN_QUADRA_ESPORTES_COBERTA,IN_QUADRA_ESPORTES_DESCOBERTA,IN_REFEITORIO,IN_SALA_ATELIE_ARTES,IN_SALA_MUSICA_CORAL,IN_SALA_ESTUDIO_DANCA,IN_SALA_MULTIUSO,IN_SALA_ESTUDIO_GRAVACAO,IN_SALA_OFICINAS_EDUC_PROF,IN_SALA_DIRETORIA,IN_SALA_LEITURA,IN_SALA_PROFESSOR,IN_SALA_REPOUSO_ALUNO,IN_SECRETARIA,IN_SALA_ATENDIMENTO_ESPECIAL,IN_TERREIRAO,IN_VIVEIRO,IN_DEPENDENCIAS_OUTRAS,IN_ACESSIBILIDADE_CORRIMAO,IN_ACESSIBILIDADE_ELEVADOR,IN_ACESSIBILIDADE_PISOS_TATEIS,IN_ACESSIBILIDADE_VAO_LIVRE,IN_ACESSIBILIDADE_RAMPAS,IN_ACESSIBILIDADE_SINAL_SONORO,IN_ACESSIBILIDADE_SINAL_TATIL,IN_ACESSIBILIDADE_SINAL_VISUAL,IN_ACESSIBILIDADE_INEXISTENTE,IN_ACESSIBILIDADE_SINALIZACAO,QT_SALAS_UTILIZADAS_DENTRO,QT_SALAS_UTILIZADAS_FORA,QT_SALAS_UTILIZADAS,QT_SALAS_UTILIZA_CLIMATIZADAS,QT_SALAS_UTILIZADAS_ACESSIVEIS,QT_SALAS_LEITURA,IN_EQUIP_PARABOLICA,IN_COMPUTADOR,IN_EQUIP_COPIADORA,IN_EQUIP_IMPRESSORA,IN_EQUIP_IMPRESSORA_MULT,IN_EQUIP_SCANNER,IN_EQUIP_NENHUM,IN_EQUIP_DVD,QT_EQUIP_DVD,IN_EQUIP_SOM,QT_EQUIP_SOM,IN_EQUIP_TV,QT_EQUIP_TV,IN_EQUIP_LOUSA_DIGITAL,QT_EQUIP_LOUSA_DIGITAL,IN_EQUIP_MULTIMIDIA,QT_EQUIP_MULTIMIDIA,IN_DESKTOP_ALUNO,QT_DESKTOP_ALUNO,IN_COMP_PORTAT

<a id="linhas"></a>
### Linhas — recorte geográfico e administrativo

Filtra o dataset consolidado para o recorte de interesse deste estudo de caso: escolas **municipais**, **em atividade**, no município de **Salvador/BA** (ver `Material e Métodos` do Projeto de Pesquisa — estudo de caso com dados de um município).


In [3]:
# ------------------------------------------------------------
# Filtros aplicados (nesta ordem): UF = BA, município = Salvador,
# TP_DEPENDENCIA = 3 (municipal), TP_SITUACAO_FUNCIONAMENTO = 1 (em atividade).
# Ajustar estes filtros altera diretamente o recorte de todo o notebook
# a partir daqui — não altera, porém, os CSVs de ENTRADA.
# ------------------------------------------------------------
# Limpeza dos dados
# Remover linhas completamente em branco (que podem existir no final da planilha)
df_censo = df_censo.dropna(how='all')

# Deixar somente as linhas que são de um determinado 
# estado, município, tipo de dependência (3= 'municipal') e situação de funcionamento (1= 'em atividade') 
df_censo= df_censo[df_censo['SG_UF'] == 'BA']
df_censo= df_censo[df_censo['NO_MUNICIPIO'] == 'Salvador']
df_censo= df_censo[df_censo['TP_DEPENDENCIA'] == 3]
df_censo= df_censo[df_censo['TP_SITUACAO_FUNCIONAMENTO'] == 1]


# Remover linhas duplicadas
print("\nVerificação de linhas duplicadas:")
print(df_censo.duplicated().sum())
df_censo = df_censo.drop_duplicates()
#print("\nLinhas duplicadas após remoção:")
#print(df_censo.duplicated().sum())

# Resetar os índices
df_censo = df_censo.reset_index(drop=True)


# Visualizar últimas linhas
display(df_censo.tail())


# Verificar se a linha com alimento vazio/none ainda existe
print(f"Processamento concluído. DataFrame final com {len(df_censo)} linhas.")



Verificação de linhas duplicadas:
0


,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,NO_REGIAO_GEOG_IMED,CO_REGIAO_GEOG_IMED,NO_MESORREGIAO,CO_MESORREGIAO,NO_MICRORREGIAO,CO_MICRORREGIAO,NO_DISTRITO,CO_DISTRITO,NO_REGIAO_ADMINISTRATIVA,CO_REGIAO_ADMINISTRATIVA,NO_ENTIDADE,CO_ENTIDADE,TP_DEPENDENCIA,TP_CATEGORIA_ESCOLA_PRIVADA,TP_LOCALIZACAO,TP_LOCALIZACAO_DIFERENCIADA,DS_ENDERECO,NU_ENDERECO,DS_COMPLEMENTO,NO_BAIRRO,CO_CEP,NU_DDD,NU_TELEFONE,LATITUDE,LONGITUDE,TP_SITUACAO_FUNCIONAMENTO,CO_ORGAO_REGIONAL,DT_ANO_LETIVO_INICIO,DT_ANO_LETIVO_TERMINO,IN_VINCULO_SECRETARIA_EDUCACAO,IN_VINCULO_SEGURANCA_PUBLICA,IN_VINCULO_SECRETARIA_SAUDE,IN_VINCULO_OUTRO_ORGAO,IN_PODER_PUBLICO_PARCERIA,TP_PODER_PUBLICO_PARCERIA,IN_FORMA_CONT_TERMO_COLABORA,IN_FORMA_CONT_TERMO_FOMENTO,IN_FORMA_CONT_ACORDO_COOP,IN_FORMA_CONT_PRESTACAO_SERV,IN_FORMA_CONT_COOP_TEC_FIN,IN_FORMA_CONT_CONSORCIO_PUB,IN_FORMA_CONT_MU_TERMO_COLAB,IN_FORMA_CONT_MU_TERMO_FOMENTO,IN_FORMA_CONT_MU_ACORDO_COOP,IN_FORMA_CONT_MU_PREST_SERV,IN_FORMA_CONT_MU_COOP_TEC_FIN,IN_FORMA_CONT_MU_CONSORCIO_PUB,IN_FORMA_CONT_ES_TERMO_COLAB,IN_FORMA_CONT_ES_TERMO_FOMENTO,IN_FORMA_CONT_ES_ACORDO_COOP,IN_FORMA_CONT_ES_PREST_SERV,IN_FORMA_CONT_ES_COOP_TEC_FIN,IN_FORMA_CONT_ES_CONSORCIO_PUB,IN_MANT_ESCOLA_PRIVADA_EMP,IN_MANT_ESCOLA_PRIVADA_ONG,IN_MANT_ESCOLA_PRIVADA_OSCIP,IN_MANT_ESCOLA_PRIV_ONG_OSCIP,IN_MANT_ESCOLA_PRIVADA_SIND,IN_MANT_ESCOLA_PRIVADA_SIST_S,IN_MANT_ESCOLA_PRIVADA_S_FINS,NU_CNPJ_ESCOLA_PRIVADA,NU_CNPJ_MANTENEDORA,TP_REGULAMENTACAO,TP_RESPONSAVEL_REGULAMENTACAO,CO_ESCOLA_SEDE_VINCULADA,CO_IES_OFERTANTE,IN_LOCAL_FUNC_PREDIO_ESCOLAR,TP_OCUPACAO_PREDIO_ESCOLAR,IN_LOCAL_FUNC_SOCIOEDUCATIVO,IN_LOCAL_FUNC_UNID_PRISIONAL,IN_LOCAL_FUNC_PRISIONAL_SOCIO,IN_LOCAL_FUNC_GALPAO,TP_OCUPACAO_GALPAO,IN_LOCAL_FUNC_SALAS_OUTRA_ESC,IN_LOCAL_FUNC_OUTROS,IN_PREDIO_COMPARTILHADO,IN_AGUA_POTAVEL,IN_AGUA_REDE_PUBLICA,IN_AGUA_POCO_ARTESIANO,IN_AGUA_CACIMBA,IN_AGUA_FONTE_RIO,IN_AGUA_INEXISTENTE,IN_AGUA_CARRO_PIPA,IN_ENERGIA_REDE_PUBLICA,IN_ENERGIA_GERADOR_FOSSIL,IN_ENERGIA_RENOVAVEL,IN_ENERGIA_INEXISTENTE,IN_ESGOTO_REDE_PUBLICA,IN_ESGOTO_FOSSA_SEPTICA,IN_ESGOTO_FOSSA_COMUM,IN_ESGOTO_FOSSA,IN_ESGOTO_INEXISTENTE,IN_LIXO_SERVICO_COLETA,IN_LIXO_QUEIMA,IN_LIXO_ENTERRA,IN_LIXO_DESTINO_FINAL_PUBLICO,IN_LIXO_DESCARTA_OUTRA_AREA,IN_TRATAMENTO_LIXO_SEPARACAO,IN_TRATAMENTO_LIXO_REUTILIZA,IN_TRATAMENTO_LIXO_RECICLAGEM,IN_TRATAMENTO_LIXO_INEXISTENTE,IN_ALMOXARIFADO,IN_AREA_VERDE,IN_AREA_PLANTIO,IN_AUDITORIO,IN_BANHEIRO,IN_BANHEIRO_EI,IN_BANHEIRO_PNE,IN_BANHEIRO_FUNCIONARIOS,IN_BANHEIRO_CHUVEIRO,IN_BIBLIOTECA,IN_BIBLIOTECA_SALA_LEITURA,IN_COZINHA,IN_DESPENSA,IN_DORMITORIO_ALUNO,IN_DORMITORIO_PROFESSOR,IN_LABORATORIO_CIENCIAS,IN_LABORATORIO_INFORMATICA,IN_LABORATORIO_EDUC_PROF,IN_PATIO_COBERTO,IN_PATIO_DESCOBERTO,IN_PARQUE_INFANTIL,IN_PISCINA,IN_QUADRA_ESPORTES,IN_QUADRA_ESPORTES_COBERTA,IN_QUADRA_ESPORTES_DESCOBERTA,IN_REFEITORIO,IN_SALA_ATELIE_ARTES,IN_SALA_MUSICA_CORAL,IN_SALA_ESTUDIO_DANCA,IN_SALA_MULTIUSO,IN_SALA_ESTUDIO_GRAVACAO,IN_SALA_OFICINAS_EDUC_PROF,IN_SALA_DIRETORIA,IN_SALA_LEITURA,IN_SALA_PROFESSOR,IN_SALA_REPOUSO_ALUNO,IN_SECRETARIA,IN_SALA_ATENDIMENTO_ESPECIAL,IN_TERREIRAO,IN_VIVEIRO,IN_DEPENDENCIAS_OUTRAS,IN_ACESSIBILIDADE_CORRIMAO,IN_ACESSIBILIDADE_ELEVADOR,IN_ACESSIBILIDADE_PISOS_TATEIS,IN_ACESSIBILIDADE_VAO_LIVRE,IN_ACESSIBILIDADE_RAMPAS,IN_ACESSIBILIDADE_SINAL_SONORO,IN_ACESSIBILIDADE_SINAL_TATIL,IN_ACESSIBILIDADE_SINAL_VISUAL,IN_ACESSIBILIDADE_INEXISTENTE,IN_ACESSIBILIDADE_SINALIZACAO,QT_SALAS_UTILIZADAS_DENTRO,QT_SALAS_UTILIZADAS_FORA,QT_SALAS_UTILIZADAS,QT_SALAS_UTILIZA_CLIMATIZADAS,QT_SALAS_UTILIZADAS_ACESSIVEIS,QT_SALAS_LEITURA,IN_EQUIP_PARABOLICA,IN_COMPUTADOR,IN_EQUIP_COPIADORA,IN_EQUIP_IMPRESSORA,IN_EQUIP_IMPRESSORA_MULT,IN_EQUIP_SCANNER,IN_EQUIP_NENHUM,IN_EQUIP_DVD,QT_EQUIP_DVD,IN_EQUIP_SOM,QT_EQUIP_SOM,IN_EQUIP_TV,QT_EQUIP_TV,IN_EQUIP_LOUSA_DIGITAL,QT_EQUIP_LOUSA_DIGITAL,IN_EQUIP_MULTIMIDIA,QT_EQUIP_MULTIMIDIA,IN_DESKTOP_ALUNO,QT_DESKTOP_ALUNO,IN_COMP_PORTAT

Processamento concluído. DataFrame final com 414 linhas.


<a id="colunas"></a>
## Limpeza e tratamento dos dados
### Colunas


In [4]:
# ------------------------------------------------------------
# Padroniza os nomes de todas as colunas do DataFrame (remove acentos,
# converte para maiúsculas, espaços -> "_"). Não afeta os CSVs de
# entrada (que permanecem intactos em disco); afeta apenas os nomes
# de coluna em memória, que serão usados dali em diante.
# ------------------------------------------------------------
# Padronização dos nomes das colunas

# Função para remover acentos
def remover_acentos(texto):
    if not isinstance(texto, str):
        return texto
    return ''.join(
        char for char in unicodedata.normalize('NFD', texto)
        if unicodedata.category(char) != 'Mn'
    )

# Remover acentos e caracteres especiais com tratamento de nulos
df_censo.columns = [
    remover_acentos(str(c)).strip().lower() if c is not None else "coluna_vazia" 
    for c in df_censo.columns
]

# Remover espaços extras
# Converter para maiusculas
# Substituir espaços por "_"

df_censo.columns = (
    df_censo.columns
      .str.strip()
      .str.upper()
      .str.replace(" ", "_")
      .str.replace("(", "", regex=False)
      .str.replace(")", "", regex=False)
      .str.replace("/", "_")
)

print("\nColunas padronizadas:")
print(df_censo.columns.tolist())



Colunas padronizadas:
['NU_ANO_CENSO', 'NO_REGIAO', 'CO_REGIAO', 'NO_UF', 'SG_UF', 'CO_UF', 'NO_MUNICIPIO', 'CO_MUNICIPIO', 'NO_REGIAO_GEOG_INTERM', 'CO_REGIAO_GEOG_INTERM', 'NO_REGIAO_GEOG_IMED', 'CO_REGIAO_GEOG_IMED', 'NO_MESORREGIAO', 'CO_MESORREGIAO', 'NO_MICRORREGIAO', 'CO_MICRORREGIAO', 'NO_DISTRITO', 'CO_DISTRITO', 'NO_REGIAO_ADMINISTRATIVA', 'CO_REGIAO_ADMINISTRATIVA', 'NO_ENTIDADE', 'CO_ENTIDADE', 'TP_DEPENDENCIA', 'TP_CATEGORIA_ESCOLA_PRIVADA', 'TP_LOCALIZACAO', 'TP_LOCALIZACAO_DIFERENCIADA', 'DS_ENDERECO', 'NU_ENDERECO', 'DS_COMPLEMENTO', 'NO_BAIRRO', 'CO_CEP', 'NU_DDD', 'NU_TELEFONE', 'LATITUDE', 'LONGITUDE', 'TP_SITUACAO_FUNCIONAMENTO', 'CO_ORGAO_REGIONAL', 'DT_ANO_LETIVO_INICIO', 'DT_ANO_LETIVO_TERMINO', 'IN_VINCULO_SECRETARIA_EDUCACAO', 'IN_VINCULO_SEGURANCA_PUBLICA', 'IN_VINCULO_SECRETARIA_SAUDE', 'IN_VINCULO_OUTRO_ORGAO', 'IN_PODER_PUBLICO_PARCERIA', 'TP_PODER_PUBLICO_PARCERIA', 'IN_FORMA_CONT_TERMO_COLABORA', 'IN_FORMA_CONT_TERMO_FOMENTO', 'IN_FORMA_CONT_ACORDO_COOP'

In [5]:
# ------------------------------------------------------------
# Reduz o DataFrame às colunas efetivamente usadas no restante do
# notebook (repasse financeiro + demanda nutricional) e exporta o
# resultado em `dataset_censo_salvador.csv`.
# SAÍDA: ./data/output/dataset_censo_salvador.csv
# Qualquer coluna nova necessária para etapas futuras (ex.: modelagem
# preditiva) deve ser adicionada à lista `colunas_para_manter` ANTES
# de gerar a versão final para o TCC, para não exigir reprocessar o
# Censo mais de uma vez.
# ------------------------------------------------------------
# Ajustes nas COLUNAS

# Deixar somente estas colunas
colunas_para_manter = [
    'SG_UF',
    'NO_MUNICIPIO',
    'NO_ENTIDADE',
    'CO_ENTIDADE',
    'TP_DEPENDENCIA',
    'TP_LOCALIZACAO',
    'TP_LOCALIZACAO_DIFERENCIADA',
    'TP_SITUACAO_FUNCIONAMENTO',
    'IN_CONVENIADA_PP',
    'IN_MEDIACAO_PRESENCIAL',
    'IN_TEMPO_INTEGRAL', 'IN_AEE', 'IN_EDUCACAO_ESPECIAL', 'IN_EDUCACAO_INDIGENA', 
    'IN_ATIVIDADE_COMPLEMENTAR', 'IN_ATENDIMENTO_ESPECIALIZADO',
    'TP_TIPO_ATENDIMENTO_TURMA', 'TP_TIPO_ATENDIMENTO', 'TP_LOCALIZACAO_TURMA', 
    'QT_MAT_BAS', 'QT_MAT_INF', 'QT_MAT_INF_CRE', 'QT_MAT_INF_PRE', 'QT_MAT_FUND', 
    'QT_MAT_FUND_AI', 'QT_MAT_FUND_AF', 'QT_MAT_MED', 'QT_MAT_PROF', 'QT_MAT_EJA', 
    'QT_MAT_ESP', 'QT_MAT_ESP_CC', 'QT_MAT_ESP_CE', 'QT_MAT_BAS_D', 'QT_MAT_BAS_ND', 
    'QT_MAT_BAS_EAD', 'QT_MAT_PROF_TEC', 'QT_MAT_PROF_TEC_CONC', 'QT_MAT_EJA_FUND', 
    'QT_MAT_EJA_MED', 'QT_MAT_BAS_INT', 'QT_MAT_INF_INT', 'QT_MAT_INF_CRE_INT', 
    'QT_MAT_INF_PRE_INT', 'QT_MAT_FUND_INT', 'QT_MAT_MED_INT',
    # Acrescentadas p/ demanda nutricional por etapa (tempo integral granular)
    'QT_MAT_FUND_AI_INT', 'QT_MAT_FUND_AF_INT', 'QT_MAT_EJA_INT',
    # Acrescentadas p/ AEE em contraturno por etapa (refeição adicional)
    'QT_MAT_ESP_CE_INF_CRE', 'QT_MAT_ESP_CE_INF_PRE', 'QT_MAT_ESP_CE_FUND_AI',
    'QT_MAT_ESP_CE_FUND_AF', 'QT_MAT_ESP_CE_MED', 'QT_MAT_ESP_CE_EJA'
]

# 'IN_QUILOMBOLA', 'TP_ETAPA_ENSINO', 'TP_MODALIDADE_ENSINO' e 'TP_SITUACAO_MATRICULA' não foram encontrado no DataFrame, 
# então não será mantido

colunas_existentes = [c for c in colunas_para_manter if c in df_censo.columns]
if colunas_existentes:
    df_censo = df_censo[colunas_existentes]

print("\nColunas atuais:")
print(df_censo.columns.tolist())

# Visualizar últimas linhas
display(df_censo.tail())

# Salvar o DataSet tratado

df_censo.to_csv(
    output_folder+"dataset_censo_salvador.csv",
    index=False,
    encoding='utf-8-sig'
)


print("\nDataset salvo com sucesso.")




Colunas atuais:
['SG_UF', 'NO_MUNICIPIO', 'NO_ENTIDADE', 'CO_ENTIDADE', 'TP_DEPENDENCIA', 'TP_LOCALIZACAO', 'TP_LOCALIZACAO_DIFERENCIADA', 'TP_SITUACAO_FUNCIONAMENTO', 'IN_MEDIACAO_PRESENCIAL', 'IN_EDUCACAO_INDIGENA', 'QT_MAT_BAS', 'QT_MAT_INF', 'QT_MAT_INF_CRE', 'QT_MAT_INF_PRE', 'QT_MAT_FUND', 'QT_MAT_FUND_AI', 'QT_MAT_FUND_AF', 'QT_MAT_MED', 'QT_MAT_PROF', 'QT_MAT_EJA', 'QT_MAT_ESP', 'QT_MAT_ESP_CC', 'QT_MAT_ESP_CE', 'QT_MAT_BAS_D', 'QT_MAT_BAS_ND', 'QT_MAT_BAS_EAD', 'QT_MAT_PROF_TEC', 'QT_MAT_PROF_TEC_CONC', 'QT_MAT_EJA_FUND', 'QT_MAT_EJA_MED', 'QT_MAT_BAS_INT', 'QT_MAT_INF_INT', 'QT_MAT_INF_CRE_INT', 'QT_MAT_INF_PRE_INT', 'QT_MAT_FUND_INT', 'QT_MAT_MED_INT', 'QT_MAT_FUND_AI_INT', 'QT_MAT_FUND_AF_INT', 'QT_MAT_EJA_INT', 'QT_MAT_ESP_CE_INF_CRE', 'QT_MAT_ESP_CE_INF_PRE', 'QT_MAT_ESP_CE_FUND_AI', 'QT_MAT_ESP_CE_FUND_AF', 'QT_MAT_ESP_CE_MED', 'QT_MAT_ESP_CE_EJA']


,SG_UF,NO_MUNICIPIO,NO_ENTIDADE,CO_ENTIDADE,TP_DEPENDENCIA,TP_LOCALIZACAO,TP_LOCALIZACAO_DIFERENCIADA,TP_SITUACAO_FUNCIONAMENTO,IN_MEDIACAO_PRESENCIAL,IN_EDUCACAO_INDIGENA,QT_MAT_BAS,QT_MAT_INF,QT_MAT_INF_CRE,QT_MAT_INF_PRE,QT_MAT_FUND,QT_MAT_FUND_AI,QT_MAT_FUND_AF,QT_MAT_MED,QT_MAT_PROF,QT_MAT_EJA,QT_MAT_ESP,QT_MAT_ESP_CC,QT_MAT_ESP_CE,QT_MAT_BAS_D,QT_MAT_BAS_ND,QT_MAT_BAS_EAD,QT_MAT_PROF_TEC,QT_MAT_PROF_TEC_CONC,QT_MAT_EJA_FUND,QT_MAT_EJA_MED,QT_MAT_BAS_INT,QT_MAT_INF_INT,QT_MAT_INF_CRE_INT,QT_MAT_INF_PRE_INT,QT_MAT_FUND_INT,QT_MAT_MED_INT,QT_MAT_FUND_AI_INT,QT_MAT_FUND_AF_INT,QT_MAT_EJA_INT,QT_MAT_ESP_CE_INF_CRE,QT_MAT_ESP_CE_INF_PRE,QT_MAT_ESP_CE_FUND_AI,QT_MAT_ESP_CE_FUND_AF,QT_MAT_ESP_CE_MED,QT_MAT_ESP_CE_EJA
409,BA,Salvador,ESCOLA MUNICIPAL RADIALISTA RAIMUNDO VARELLA F...,29488729,3,1,0.0,1,1.0,0.0,273.0,0.0,0.0,0.0,273.0,273.0,0.0,0.0,0.0,0.0,21.0,21.0,0.0,273.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
410,BA,Salvador,CENTRO MUNICIPAL DE EDUCACAO INFANTIL DONA MAR...,29488842,3,1,0.0,1,1.0,0.0,382.0,234.0,117.0,117.0,148.0,148.0,0.0,0.0,0.0,0.0,40.0,40.0,0.0,382.0,0.0,0.0,0.0,0.0,0.0,0.0,382.0,234.0,117.0,117.0,148.0,0.0,148.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
411,BA,Salvador,ESCOLA MUNICIPAL PROFESSORA OLGA METTIG,29534801,3,1,0.0,1,1.0,0.0,540.0,0.0,0.0,0.0,540.0,0.0,540.0,0.0,0.0,0.0,40.0,40.0,0.0,540.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,0.0,15.0,0.0,0.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
412,BA,Salvador,ESCOLA MUNICIPAL MANOEL HENRIQUE DA SILVA BARR...,29854750,3,1,0.0,1,1.0,0.0,301.0,0.0,0.0,0.0,301.0,0.0,301.0,0.0,0.0,0.0,25.0,25.0,0.0,301.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
413,BA,Salvador,ESCOLA MUNICIPAL JARDIM SANTO INACIO,29994756,3,1,0.0,1,1.0,0.0,679.0,0.0,0.0,0.0,554.0,275.0,279.0,0.0,0.0,125.0,44.0,44.0,0.0,554.0,0.0,0.0,0.0,0.0,125.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Dataset salvo com sucesso.


<a id="dados-faltantes"></a>
### Colunas — tratamento dos dados
Dados faltantes e imputação de dados

Esta seção verifica a completude das colunas mantidas (ver [seção anterior](#colunas)) **antes** de seguir para o cálculo de repasse e de demanda nutricional. Nenhuma imputação é realizada aqui: o objetivo é apenas diagnosticar. Caso o diagnóstico revele lacunas relevantes, a estratégia de tratamento (descarte vs. imputação) deve ser decidida e documentada explicitamente, conforme o roteiro de ETL.


In [6]:
# ------------------------------------------------------------
# DIAGNÓSTICO DE DADOS FALTANTES
# Esta célula é apenas de leitura/diagnóstico: NÃO gera nem sobrescreve
# nenhum arquivo CSV. 
# O código abaixo cobre o item "Verifique dados faltantes" do roteiro
# de ETL (ver seção "Roteiro de Ingestão e ETL").
# ------------------------------------------------------------

print("Valores ausentes por coluna (contagem):")
display(df_censo.isna().sum().sort_values(ascending=False))

print("\nValores ausentes por coluna (% do total de linhas):")
display((df_censo.isna().mean() * 100).round(2).sort_values(ascending=False))

# TODO (validar com a orientação antes do TCC final):
# Caso alguma coluna usada no cálculo de repasse ou de demanda
# nutricional (ex.: QT_MAT_*) apresente valores ausentes, o notebook
# já trata isso pontualmente com `.fillna(0)` nos blocos de cálculo
# (repasse e demanda nutricional). Decidir e registrar aqui, de forma
# explícita, se `fillna(0)` é sempre a estratégia correta ou se algum
# caso exige descarte/imputação por outra fonte.


Valores ausentes por coluna (contagem):


QT_MAT_INF                     2
QT_MAT_INF_CRE                 2
QT_MAT_BAS                     2
QT_MAT_ESP_CE                  2
QT_MAT_ESP_CC                  2
QT_MAT_ESP                     2
QT_MAT_EJA                     2
QT_MAT_PROF                    2
QT_MAT_MED                     2
QT_MAT_FUND_AF                 2
QT_MAT_FUND_AI                 2
QT_MAT_FUND                    2
QT_MAT_INF_PRE                 2
QT_MAT_EJA_MED                 2
QT_MAT_BAS_INT                 2
QT_MAT_INF_INT                 2
QT_MAT_INF_CRE_INT             2
QT_MAT_INF_PRE_INT             2
QT_MAT_FUND_INT                2
QT_MAT_MED_INT                 2
QT_MAT_FUND_AI_INT             2
QT_MAT_FUND_AF_INT             2
QT_MAT_EJA_INT                 2
QT_MAT_BAS_D                   2
QT_MAT_BAS_ND                  2
QT_MAT_BAS_EAD                 2
QT_MAT_PROF_TEC                2
QT_MAT_PROF_TEC_CONC           2
QT_MAT_EJA_FUND                2
QT_MAT_ESP_CE_FUND_AI          2
QT_MAT_ESP


Valores ausentes por coluna (% do total de linhas):


QT_MAT_INF                     0.48
QT_MAT_INF_CRE                 0.48
QT_MAT_BAS                     0.48
QT_MAT_ESP_CE                  0.48
QT_MAT_ESP_CC                  0.48
QT_MAT_ESP                     0.48
QT_MAT_EJA                     0.48
QT_MAT_PROF                    0.48
QT_MAT_MED                     0.48
QT_MAT_FUND_AF                 0.48
QT_MAT_FUND_AI                 0.48
QT_MAT_FUND                    0.48
QT_MAT_INF_PRE                 0.48
QT_MAT_EJA_MED                 0.48
QT_MAT_BAS_INT                 0.48
QT_MAT_INF_INT                 0.48
QT_MAT_INF_CRE_INT             0.48
QT_MAT_INF_PRE_INT             0.48
QT_MAT_FUND_INT                0.48
QT_MAT_MED_INT                 0.48
QT_MAT_FUND_AI_INT             0.48
QT_MAT_FUND_AF_INT             0.48
QT_MAT_EJA_INT                 0.48
QT_MAT_BAS_D                   0.48
QT_MAT_BAS_ND                  0.48
QT_MAT_BAS_EAD                 0.48
QT_MAT_PROF_TEC                0.48
QT_MAT_PROF_TEC_CONC        

<a id="regras-repasse"></a>
## Regras de repasse financeiro do PNAE

RESOLUÇÃO CD/FNDE Nº 1, DE 18 DE FEVEREIRO DE 2026

a) R$ 0,57 (cinquenta e sete centavos) para os estudantes matriculados no ensino fundamental,no ensino médio e na Educação de Jovens e Adultos - EJA;
- repasse = (QT_MAT_FUND+QT_MAT_MED+QT_MAT_EJA)×0,57×200

b) R$ 0,82 (oitenta e dois centavos) para estudantes matriculados na pré-escola, exceto os matriculados em escolas localizadas em áreas indígenas, remanescentes de quilombos e demais povos e comunidades tradicionais;
- Pre-escola comum=QT_MAT_INF_PRE−Pre indıgena/quilombola
- repasse = Pre-escola comum×0,82×200

c) R$ 0,98 (noventa e oito centavos) para os estudantes matriculados em escolas de educação básica localizadas em áreas indígenas, remanescentes de quilombos e demais povos e comunidades tradicionais;
- repasse = QT_MAT_DIFERENCIADA×0,98×200  
Obs: (O Censo não possui um único QT_MAT_*_INDIGENA)

d) R$ 1,57 (um real e cinquenta e sete centavos) para os estudantes matriculados em escolas de tempo integral com permanência mínima de sete horas na escola ou em atividades escolares, de acordo com o Censo Escolar do Inep; e
- repasse = QT_MAT_BAS_INT×1,57×200  --> Simplificada
Obs: Nem toda matrícula integral recebe esse valor automaticamente. A resolução fala: permanência mínima de 7 horas. 

e) R$ 1,57 (um real e cinquenta e sete centavos) para os estudantes matriculados em creches,inclusive as localizadas em áreas indígenas, remanescentes de quilombos e demais povos e comunidades tradicionais;
- repasse = QT_MAT_INF_CRE×1,57×200

.................................................................................................

IV - para os estudantes contemplados no Programa de Fomento às Escolas de Ensino Médio em Tempo Integral, haverá complementação financeira de forma a totalizar o valor per capita de R$ 2,93 (dois reais e noventa e três centavos);
- diferenca_complementacao = 2,93−1,57=1,36
- repasse_EMTI = QT_EMTI×1,36×200 como QT_EMTI não existe, aproximação possível QT_MAT_MED_INT, mas isso superestima, porque nem todo médio integral participa do EMTI.

V - para os estudantes que frequentam, no contraturno, o Atendimento Educacional Especializado - AEE, o valor per capita será de R$ 0,78 (setenta e oito centavos);
- repasse = QT_AEE×0,78×200
- Não existe QT_AEE - QT_MAT_ESP não representa necessariamente AEE no contraturno. O correto é identificar:
matrícula ativa com AEE complementar, mas não tem nesta base consolidada. 

Repasse Total= (Fundamental+Medio+EJA)×0,57×200
+(Pre-escola comum)×0,82×200
+(Indıgena/Quilombola)×0,98×200
+(Tempo Integral)×1,57×200
+(Creche)×1,57×200
+(EMTI)×1,36×200
+(AEE)×0,78×200

In [7]:
# ------------------------------------------------------------
# ENTRADA (em memória): df_censo (saída da seção "Colunas")
# SAÍDA: ./data/output/resumo_repasse_pnae_amostra.csv
#
# ⚠️ PONTO DE ATENÇÃO (revisar antes da entrega final do TCC):
#
# Há dois blocos marcados "VERIFICAR" abaixo
# (`matricula_complementacao` e `matricula_aee_contraturno`) que ainda
# não tiveram a regra de elegibilidade confirmada com a orientação.
# ------------------------------------------------------------

# CONFIGURAÇÕES

dias_letivos = 200

vl_diario_fund_med_eja = 0.57
vl_diario_infpre_comum = 0.82
vl_diario_diferenciada = 0.98
vl_diario_integral = 1.57
vl_diario_creche = 1.57
vl_dif_complementacao = 2.93 - vl_diario_integral
vl_diario_aee_contraturno = 0.78





# Vamos trabalhar com 100% das escolas públicas municipais ativas

# Remove possíveis duplicidades de escola
escolas_unicas = df_censo['CO_ENTIDADE'].drop_duplicates()

# Define quantidade da amostra (1 = 100% das escolas) 
# (lógica mantida caso deseje-se alterar para amostra representativa em algum momento)
qtd_escolas_amostra = int(len(escolas_unicas) * 1)

# Sorteio reprodutível
np.random.seed(42)

escolas_amostra = np.random.choice(
    escolas_unicas,
    size=qtd_escolas_amostra,
    replace=False
)

# Filtra somente as escolas da amostra
df_censo_amostra = df_censo[df_censo['CO_ENTIDADE'].isin(escolas_amostra)].copy()

print(f'Total de escolas no dataset: {len(escolas_unicas)}')
print(f'Total de escolas na amostra: {len(escolas_amostra)}')


# ============================================================
# CÁLCULO DAS MATRÍCULAS POR MODALIDADE
# Contagem dos alunos para cada tipo de repasse, 
# considerando as regras de elegibilidade e evitando dupla contagem
# ============================================================


# a) FUNDAMENTAL + MÉDIO + EJA
# Evita dupla contagem com Integral 
# (subtraído QT_MAT_FUND_INT e QT_MAT_MED_INT porque alunos 
# de Integral não podem receber repasse para Fundamental ou Médio, 
# para evitar dupla contagem)
# Os alunos de escolas em localização difrenciada são computados na 
# modalidade diferenciada que tem valor maior
df_censo_amostra['matricula_fund_med_eja'] = np.where(
    ~df_censo_amostra['TP_LOCALIZACAO_DIFERENCIADA'].isin([2, 3, 8]),
    (
        df_censo_amostra['QT_MAT_FUND'].fillna(0) +
        df_censo_amostra['QT_MAT_MED'].fillna(0) +
        df_censo_amostra['QT_MAT_EJA'].fillna(0) -
        df_censo_amostra['QT_MAT_FUND_INT'].fillna(0) -
        df_censo_amostra['QT_MAT_MED_INT'].fillna(0)
    ),
    0
)


# b) INFANTIL / PRÉ-ESCOLA COMUM
# Exclui escolas indígenas
# VERIFICAR COMO CHECAR SE A ESCOLA É QUILOMBOLA TAMBÉM 
df_censo_amostra['matricula_infpre_comum'] = np.where(
    df_censo_amostra['TP_LOCALIZACAO_DIFERENCIADA'].isin([2, 3, 8]),
    0,
    df_censo_amostra['QT_MAT_INF_PRE'].fillna(0) - 
    df_censo_amostra['QT_MAT_INF_PRE_INT'].fillna(0)
)


# d) EDUCAÇÃO INTEGRAL
# Calcula antes para permitir abatimento em diferenciada
# Evita dupla contagem com creche integral 
# (NÃO USADO df_censo['QT_MAT_INF_CRE_INT'] porque alunos de Creche 
# não podem receber repasse para Integral, para evitar dupla contagem)
df_censo_amostra['matricula_integral'] = (
    df_censo_amostra['QT_MAT_INF_PRE_INT'].fillna(0) +
    df_censo_amostra['QT_MAT_FUND_INT'].fillna(0) +
    df_censo_amostra['QT_MAT_MED_INT'].fillna(0)
)

# Evita valores negativos
df_censo_amostra['matricula_integral'] = (
    df_censo_amostra['matricula_integral'].clip(lower=0)
)


# e) CRECHE
# Calcula antes para permitir abatimento em diferenciada
df_censo_amostra['matricula_creche'] = (
    df_censo_amostra['QT_MAT_INF_CRE'].fillna(0) # creche integral e não integral
)


# c) LOCALIZAÇÃO DIFERENCIADA
# Abate integral e creche para evitar dupla contagem
# pois diferenciada possui menor valor per capita que integral e creche
df_censo_amostra['matricula_diferenciada'] = np.where(
    df_censo_amostra['TP_LOCALIZACAO_DIFERENCIADA'].isin([2, 3, 8]),
    (
        df_censo_amostra['QT_MAT_BAS'].fillna(0)
        - df_censo_amostra['QT_MAT_INF_PRE_INT'].fillna(0) 
        - df_censo_amostra['QT_MAT_FUND_INT'].fillna(0) 
        - df_censo_amostra['QT_MAT_MED_INT'].fillna(0)
        - df_censo_amostra['QT_MAT_INF_CRE'].fillna(0)    # creche integral e não integral
    ),

    0
)

# Evita valores negativos
df_censo_amostra['matricula_diferenciada'] = (
    df_censo_amostra['matricula_diferenciada'].clip(lower=0)
)


# COMPLEMENTAÇÃO
################## VERIFICAR
df_censo_amostra['matricula_complementacao'] = (
    df_censo_amostra['QT_MAT_MED_INT'].fillna(0)
)


# AEE CONTRATURNO
################## VERIFICAR se seria QT_MAT_ESP_CE
df_censo_amostra['matricula_aee_contraturno'] = (
    df_censo_amostra['QT_MAT_ESP_CE'].fillna(0)
)

# ============================================================
# TOTAL DE MATRÍCULAS COM DIREITO AO REPASSE
# ============================================================
# 'matricula_complementacao' não foi incluída no total de matrículas com direito ao repasse,
# pois é um tipo de repasse específico para alunos de integral, então não é somada para evitar dupla contagem

modalidades = [
    'matricula_fund_med_eja',
    'matricula_infpre_comum',
    'matricula_diferenciada',
    'matricula_integral',
    'matricula_creche',
    'matricula_aee_contraturno'
]

df_censo_amostra['total_matriculas_repasse'] = (
    df_censo_amostra[modalidades].sum(axis=1)
)

# ============================================================
# CÁLCULO DOS VALORES DO REPASSE
# ============================================================

df_censo_amostra['vl_fund_med_eja'] = (
    df_censo_amostra['matricula_fund_med_eja'] *
    vl_diario_fund_med_eja *
    dias_letivos
)

df_censo_amostra['vl_infpre_comum'] = (
    df_censo_amostra['matricula_infpre_comum'] *
    vl_diario_infpre_comum *
    dias_letivos
)

df_censo_amostra['vl_diferenciada'] = (
    df_censo_amostra['matricula_diferenciada'] *
    vl_diario_diferenciada *
    dias_letivos
)

df_censo_amostra['vl_integral'] = (
    df_censo_amostra['matricula_integral'] *
    vl_diario_integral *
    dias_letivos
)

df_censo_amostra['vl_creche'] = (
    df_censo_amostra['matricula_creche'] *
    vl_diario_creche *
    dias_letivos
)

df_censo_amostra['vl_complementacao'] = (
    df_censo_amostra['matricula_complementacao'] *
    vl_dif_complementacao *
    dias_letivos
)

df_censo_amostra['vl_aee_contraturno'] = (
    df_censo_amostra['matricula_aee_contraturno'] *
    vl_diario_aee_contraturno *
    dias_letivos
)

# ============================================================
# TOTAL ANUAL POR ESCOLA
# ============================================================

valores = [
    'vl_fund_med_eja',
    'vl_infpre_comum',
    'vl_diferenciada',
    'vl_integral',
    'vl_creche',
    'vl_complementacao',
    'vl_aee_contraturno'
]

df_censo_amostra['valor_total_repasse_anual'] = (
    df_censo_amostra[valores].sum(axis=1)
)



# ============================================================
# DEMANDA DE REFEIÇÕES
# Base para Programação Linear
# ============================================================

# Refeições por aluno/dia

REF_FUND_MED_EJA      = 1
REF_INFPRE_COMUM      = 1
REF_DIFERENCIADA      = 1
REF_INTEGRAL          = 3
REF_CRECHE            = 2
REF_AEE_CONTRATURNO   = 1

# ------------------------------------------------------------
# Quantidade diária de refeições por modalidade
# ------------------------------------------------------------

df_censo_amostra['ref_dia_fund_med_eja'] = (
    df_censo_amostra['matricula_fund_med_eja']
    * REF_FUND_MED_EJA
)

df_censo_amostra['ref_dia_infpre_comum'] = (
    df_censo_amostra['matricula_infpre_comum']
    * REF_INFPRE_COMUM
)

df_censo_amostra['ref_dia_diferenciada'] = (
    df_censo_amostra['matricula_diferenciada']
    * REF_DIFERENCIADA
)

df_censo_amostra['ref_dia_integral'] = (
    df_censo_amostra['matricula_integral']
    * REF_INTEGRAL
)

df_censo_amostra['ref_dia_creche'] = (
    df_censo_amostra['matricula_creche']
    * REF_CRECHE
)

df_censo_amostra['ref_dia_aee'] = (
    df_censo_amostra['matricula_aee_contraturno']
    * REF_AEE_CONTRATURNO
)

# ------------------------------------------------------------
# Total diário de refeições da escola
# ------------------------------------------------------------

colunas_refeicoes = [
    'ref_dia_fund_med_eja',
    'ref_dia_infpre_comum',
    'ref_dia_diferenciada',
    'ref_dia_integral',
    'ref_dia_creche',
    'ref_dia_aee'
]

df_censo_amostra['total_refeicoes_dia'] = (
    df_censo_amostra[colunas_refeicoes].sum(axis=1)
)

# ------------------------------------------------------------
# Total anual de refeições
# ------------------------------------------------------------

df_censo_amostra['total_refeicoes_ano'] = (
    df_censo_amostra['total_refeicoes_dia']
    * dias_letivos
)





# ============================================================
# RESUMO POR ESCOLA
# ============================================================

colunas_resumo = [
    'CO_ENTIDADE',
    'NO_ENTIDADE',

    'matricula_fund_med_eja',
    'matricula_infpre_comum',
    'matricula_diferenciada',
    'matricula_integral',
    'matricula_creche',
    'matricula_complementacao',
    'matricula_aee_contraturno',

    'ref_dia_fund_med_eja',
    'ref_dia_infpre_comum',
    'ref_dia_diferenciada',
    'ref_dia_integral',
    'ref_dia_creche',
    'ref_dia_aee',

    'total_refeicoes_dia',
    'total_refeicoes_ano',

    'total_matriculas_repasse',
    'valor_total_repasse_anual'
]

df_resumo_escolas = df_censo_amostra[colunas_resumo].copy()

# ============================================================
# EXIBIÇÃO DOS RESULTADOS
# ============================================================

pd.set_option('display.max_columns', None)

print('\n===================================================')
print('RESUMO POR ESCOLA')
print('===================================================')

print(df_resumo_escolas.head())

# ============================================================
# TOTAIS GERAIS DA AMOSTRA
# ============================================================

print('\n===================================================')
print('TOTAIS GERAIS DA AMOSTRA')
print('===================================================')

for modalidade in modalidades:
    total_modalidade = df_censo_amostra[modalidade].sum()
    print(f'{modalidade}: {total_modalidade:,.0f}')

print('\n--------------------------------------------')

total_geral_matriculas = (
    df_censo_amostra['total_matriculas_repasse'].sum()
)

print(f'TOTAL GERAL DE MATRÍCULAS DA AMOSTRA: {total_geral_matriculas:,.0f}')

print('\n--------------------------------------------')

for valor in valores:
    total_valor = df_censo_amostra[valor].sum()
    print(f'{valor}: R$ {total_valor:,.2f}')

print('\n--------------------------------------------')

valor_total_amostra = (
    df_censo_amostra['valor_total_repasse_anual'].sum()
)

print(f'VALOR TOTAL ANUAL DO REPASSE DA AMOSTRA:')
print(f'R$ {valor_total_amostra:,.2f}')

# ============================================================
# OPCIONAL - EXPORTAR CSV
# ============================================================

df_resumo_escolas.to_csv(
    output_folder + 'resumo_repasse_pnae_amostra.csv',
    index=False,
    sep=';',
    decimal=',',
    encoding='utf-8-sig'
)

print('\nArquivo CSV com resumo da amostra exportado com sucesso.')


Total de escolas no dataset: 414
Total de escolas na amostra: 414

RESUMO POR ESCOLA
   CO_ENTIDADE                                        NO_ENTIDADE  \
0     29179319                ESCOLA MUNICIPAL CAMPINAS DE PIRAJA   
1     29179475  CENTRO MUNICIPAL DE EDUCACAO INFANTIL CASA DA ...   
2     29179670  ESCOLA MUNICIPAL CENTRO COMUNITARIO FREI LEONI...   
3     29180058  CENTRO MUNICIPAL DE EDUCACAO INFANTIL JOAQUIM ...   
4     29180066                   ESCOLA MUNICIPAL ELOYNA BARRADAS   

   matricula_fund_med_eja  matricula_infpre_comum  matricula_diferenciada  \
0                   698.0                     0.0                     0.0   
1                     0.0                    59.0                     0.0   
2                   131.0                     0.0                     0.0   
3                     0.0                     0.0                     0.0   
4                   237.0                   107.0                     0.0   

   matricula_integral  matricula_crec

<a id="demanda-nutricional"></a>
## Demanda nutricional por escola (grade para Programação Linear)

Bloco complementar ao cálculo de repasse acima. Gera `demanda_nutricional_escolas.csv` com uma linha por combinação **escola × grupo nutricional × número de refeições/dia**, já casada com os valores de referência (energia, macro e micronutrientes) dos Anexos de Valores de Referência do PNAE.

**Por que este bloco não reaproveita as colunas `matricula_*` do repasse?**
A contagem para fins de repasse FNDE segue regras de **elegibilidade financeira** (ex.: escolas de localização diferenciada são contadas à parte, com valor per capita distinto, para não gerar dupla contagem de recurso). A demanda nutricional não tem essa regra — todo aluno precisa se alimentar, independentemente de qual rubrica financia a refeição. Por isso este bloco recalcula a contagem de alunos diretamente das colunas **brutas** de matrícula por etapa (`QT_MAT_INF_CRE`, `QT_MAT_INF_PRE`, `QT_MAT_FUND_AI`, `QT_MAT_FUND_AF`, `QT_MAT_MED`, `QT_MAT_EJA`) e seus respectivos pares `_INT` (tempo integral).

**Premissas assumidas nesta versão (pela autorora do projeto):**
- **Creche:** 100% das matrículas usam a referência de *Creche (1-3 anos)* — o Censo não permite separar 7-11 meses de 1-3 anos (não existe variável de matrícula com essa granularidade etária).
- **EJA:** referência fixada em *EJA (19 a 30 anos)* — o Censo não informa idade do aluno.
- **AEE em contraturno e localização diferenciada** não geram grupo nutricional próprio nesta entrega (ficam fora do CSV). Se quiser detalhá-los depois, o Censo possui as variáveis `QT_MAT_ESP_CE_INF_CRE`, `QT_MAT_ESP_CE_INF_PRE`, `QT_MAT_ESP_CE_FUND_AI`, `QT_MAT_ESP_CE_FUND_AF`, `QT_MAT_ESP_CE_MED`, `QT_MAT_ESP_CE_EJA` — basta replicar o mesmo padrão de `GRUPOS_NUTRICIONAIS` abaixo, lembrando que esse é um *adicional* de refeição (1/dia), não substitui a refeição já contada no grupo de origem do aluno.


In [8]:
# ------------------------------------------------------------
# ENTRADA (em memória): df_censo_amostra (saída da seção "Cálculo do
# repasse financeiro" — reaproveita o mesmo recorte de escolas, mas
# recalcula a contagem de alunos a partir das colunas BRUTAS de
# matrícula, sem as regras de elegibilidade financeira; ver
# justificativa detalhada na célula markdown desta seção).
# SAÍDA: ./data/output/demanda_nutricional_escolas.csv
#   -> Este é o CSV consumido diretamente pelo modelo de Programação
#      Linear (Problema da Dieta) como restrição de demanda (RHS).
# ------------------------------------------------------------
# ============================================================
# DEMANDA NUTRICIONAL POR ESCOLA — GRADE PARA PROGRAMAÇÃO LINEAR
# ============================================================

# ------------------------------------------------------------
# TABELA DE REFERÊNCIA NUTRICIONAL — Anexos de Valores de
# Referência do PNAE
#   - Creche: inclui Cálcio, Ferro, Vit. A, Vit. C (sem Sódio)
#   - Demais etapas: inclui Sódio (sem Ca, Fe, Vit. A, Vit. C)
#   - Colunas ausentes na fonte oficial ficam como NaN; a PL deve
#     IGNORAR (não tratar como zero) os limites com NaN.
# ------------------------------------------------------------

REFERENCIA_NUTRICIONAL_PNAE = pd.DataFrame([
    # grupo_nutricional, num_refeicoes, energia_kcal,
    # proteina_min, proteina_max, lipidio_min, lipidio_max,
    # carbo_min, carbo_max, calcio, ferro, vitamina_a, vitamina_c, sodio
    ['CRECHE',       2,  304,  8,  11, 8,  12, 42,  49,  150.0, 1.0, 63.0,  4.0, np.nan],
    ['CRECHE',       3,  708,  18, 27, 20, 28, 97,  115, 350.0, 2.0, 147.0, 9.0, np.nan],

    ['PRE_ESCOLA',   1,  270,  7,  10, 8,  11, 37,  44,  np.nan, np.nan, np.nan, np.nan, 600],
    ['PRE_ESCOLA',   3,  945,  24, 35, 26, 37, 130, 154, np.nan, np.nan, np.nan, np.nan, 1400],

    ['FUND_6_10',    1,  329,  8,  12, 9,  13, 45,  53,  np.nan, np.nan, np.nan, np.nan, 600],
    ['FUND_6_10',    3,  1150, 29, 43, 32, 45, 158, 187, np.nan, np.nan, np.nan, np.nan, 1400],

    ['FUND_11_15',   1,  473,  12, 18, 13, 18, 65,  77,  np.nan, np.nan, np.nan, np.nan, 600],
    ['FUND_11_15',   3,  1656, 41, 62, 46, 64, 228, 269, np.nan, np.nan, np.nan, np.nan, 1400],

    ['ENSINO_MEDIO', 1,  543,  14, 20, 15, 21, 75,  88,  np.nan, np.nan, np.nan, np.nan, 600],
    ['ENSINO_MEDIO', 3,  1902, 48, 71, 53, 74, 262, 309, np.nan, np.nan, np.nan, np.nan, 1400],

    ['EJA',          1,  477,  12, 18, 8,  16, 66,  77,  np.nan, np.nan, np.nan, np.nan, 600],
    ['EJA',          3,  1668, 42, 63, 28, 56, 229, 271, np.nan, np.nan, np.nan, np.nan, 1400],
], columns=[
    'grupo_nutricional', 'num_refeicoes', 'energia_kcal',
    'proteina_min', 'proteina_max', 'lipidio_min', 'lipidio_max',
    'carbo_min', 'carbo_max', 'calcio', 'ferro', 'vitamina_a',
    'vitamina_c', 'sodio'
])

# ------------------------------------------------------------
# MAPA: grupo_nutricional -> colunas de matrícula no Censo
#   col_total    = todas as matrículas da etapa (já inclui integral)
#   col_integral = subconjunto em tempo integral (subset de col_total)
#   ref_comum    = nº de refeições/dia do regime NÃO-integral
#                  (regime integral usa sempre REF_INTEGRAL = 3)
# ------------------------------------------------------------

GRUPOS_NUTRICIONAIS = {
    'CRECHE':       {'col_total': 'QT_MAT_INF_CRE', 'col_integral': 'QT_MAT_INF_CRE_INT', 'ref_comum': REF_CRECHE},
    'PRE_ESCOLA':   {'col_total': 'QT_MAT_INF_PRE', 'col_integral': 'QT_MAT_INF_PRE_INT', 'ref_comum': REF_INFPRE_COMUM},
    'FUND_6_10':    {'col_total': 'QT_MAT_FUND_AI', 'col_integral': 'QT_MAT_FUND_AI_INT', 'ref_comum': REF_FUND_MED_EJA},
    'FUND_11_15':   {'col_total': 'QT_MAT_FUND_AF', 'col_integral': 'QT_MAT_FUND_AF_INT', 'ref_comum': REF_FUND_MED_EJA},
    'ENSINO_MEDIO': {'col_total': 'QT_MAT_MED',     'col_integral': 'QT_MAT_MED_INT',     'ref_comum': REF_FUND_MED_EJA},
    'EJA':          {'col_total': 'QT_MAT_EJA',     'col_integral': 'QT_MAT_EJA_INT',     'ref_comum': REF_FUND_MED_EJA},
}

colunas_necessarias = {
    c for cfg in GRUPOS_NUTRICIONAIS.values()
    for c in (cfg['col_total'], cfg['col_integral'])
}
colunas_faltantes = colunas_necessarias - set(df_censo_amostra.columns)
if colunas_faltantes:
    raise KeyError(
        f"Colunas ausentes em df_censo_amostra: {sorted(colunas_faltantes)}. "
        "Verifique se foram incluídas em 'colunas_para_manter' "
        "(bloco 'Ajustes nas COLUNAS')."
    )

# ------------------------------------------------------------
# MONTAGEM DA GRADE (formato longo): 1 linha por escola x grupo x regime
# ------------------------------------------------------------

linhas_demanda = []

for grupo, cfg in GRUPOS_NUTRICIONAIS.items():
    total = df_censo_amostra[cfg['col_total']].fillna(0)
    integral = df_censo_amostra[cfg['col_integral']].fillna(0)

    # Matrículas em regime comum = total - integral
    # (QT_MAT_*_INT é subconjunto de QT_MAT_*; clip evita negativos
    # por eventuais inconsistências de preenchimento do Censo)
    comum = (total - integral).clip(lower=0)

    base = df_censo_amostra[['CO_ENTIDADE', 'NO_ENTIDADE']].copy()

    df_comum = base.copy()
    df_comum['grupo_nutricional'] = grupo
    df_comum['num_refeicoes'] = cfg['ref_comum']
    df_comum['quantidade_alunos'] = comum
    linhas_demanda.append(df_comum)

    df_integral = base.copy()
    df_integral['grupo_nutricional'] = grupo
    df_integral['num_refeicoes'] = REF_INTEGRAL
    df_integral['quantidade_alunos'] = integral
    linhas_demanda.append(df_integral)

# ------------------------------------------------------------
# AEE EM CONTRATURNO — refeição ADICIONAL
# Não substitui a refeição já contada no grupo de origem do aluno;
# soma-se a ela (mesmo padrão usado no bloco de repasse: ref_dia_aee
# é somado independentemente das demais modalidades).
# Usa a referência nutricional de '1 refeição' do próprio grupo
# etário, pois o Anexo do PNAE não define tabela própria para o
# atendimento de AEE em contraturno.
# Creche é INTENCIONALMENTE excluída: o Anexo só tem referência de
# 2 e 3 refeições/dia para creche (não existe linha de '1 refeição'
# / 20% do VET), logo não há base oficial defensável para essa
# combinação. Ver alerta abaixo caso existam matrículas.
# ------------------------------------------------------------

GRUPOS_AEE_CONTRATURNO = {
    'PRE_ESCOLA_AEE':   'QT_MAT_ESP_CE_INF_PRE',
    'FUND_6_10_AEE':    'QT_MAT_ESP_CE_FUND_AI',
    'FUND_11_15_AEE':   'QT_MAT_ESP_CE_FUND_AF',
    'ENSINO_MEDIO_AEE': 'QT_MAT_ESP_CE_MED',
    'EJA_AEE':          'QT_MAT_ESP_CE_EJA',
}

colunas_aee_faltantes = (
    set(GRUPOS_AEE_CONTRATURNO.values()) | {'QT_MAT_ESP_CE_INF_CRE'}
) - set(df_censo_amostra.columns)
if colunas_aee_faltantes:
    raise KeyError(
        f"Colunas ausentes em df_censo_amostra: {sorted(colunas_aee_faltantes)}. "
        "Verifique se foram incluídas em 'colunas_para_manter'."
    )

# Deriva a referência nutricional do AEE a partir da linha de
# '1 refeição' de cada grupo já existente (mesma faixa etária)
ref_aee = (
    REFERENCIA_NUTRICIONAL_PNAE[
        REFERENCIA_NUTRICIONAL_PNAE['num_refeicoes'] == REF_FUND_MED_EJA
    ]
    .copy()
)
ref_aee['grupo_nutricional'] = ref_aee['grupo_nutricional'] + '_AEE'
ref_aee['num_refeicoes'] = REF_AEE_CONTRATURNO
REFERENCIA_NUTRICIONAL_PNAE = pd.concat(
    [REFERENCIA_NUTRICIONAL_PNAE, ref_aee], ignore_index=True
)

for grupo_aee, coluna in GRUPOS_AEE_CONTRATURNO.items():
    df_aee = df_censo_amostra[['CO_ENTIDADE', 'NO_ENTIDADE']].copy()
    df_aee['grupo_nutricional'] = grupo_aee
    df_aee['num_refeicoes'] = REF_AEE_CONTRATURNO
    df_aee['quantidade_alunos'] = df_censo_amostra[coluna].fillna(0)
    linhas_demanda.append(df_aee)

# Alerta: creche em AEE contraturno não tem referência oficial
qt_aee_creche = df_censo_amostra['QT_MAT_ESP_CE_INF_CRE'].fillna(0).sum()
if qt_aee_creche > 0:
    print(
        f"ALERTA: {qt_aee_creche:,.0f} matrículas de Creche em AEE "
        "contraturno (QT_MAT_ESP_CE_INF_CRE) foram encontradas, mas "
        "NÃO entraram na demanda nutricional — o Anexo do PNAE não "
        "define referência de '1 refeição' para creche. Avalie "
        "manualmente esse caso."
    )

df_demanda_nutricional = pd.concat(linhas_demanda, ignore_index=True)

# Remove combinações sem alunos (não geram demanda para a PL)
df_demanda_nutricional = (
    df_demanda_nutricional[df_demanda_nutricional['quantidade_alunos'] > 0]
    .copy()
)

# ------------------------------------------------------------
# JOIN COM A TABELA DE REFERÊNCIA NUTRICIONAL
# ------------------------------------------------------------

df_demanda_nutricional = df_demanda_nutricional.merge(
    REFERENCIA_NUTRICIONAL_PNAE,
    on=['grupo_nutricional', 'num_refeicoes'],
    how='left'
)

# Checagem de integridade: toda combinação grupo+refeições precisa
# ter encontrado referência nutricional correspondente
sem_referencia = df_demanda_nutricional[df_demanda_nutricional['energia_kcal'].isna()]
if len(sem_referencia) > 0:
    print('ALERTA: combinações sem referência nutricional encontrada:')
    print(sem_referencia[['grupo_nutricional', 'num_refeicoes']].drop_duplicates())

# ------------------------------------------------------------
# ORDENAÇÃO E EXPORTAÇÃO
# ------------------------------------------------------------

colunas_finais = [
    'CO_ENTIDADE', 'NO_ENTIDADE', 'grupo_nutricional', 'num_refeicoes',
    'quantidade_alunos', 'energia_kcal', 'proteina_min', 'proteina_max',
    'lipidio_min', 'lipidio_max', 'carbo_min', 'carbo_max',
    'calcio', 'ferro', 'vitamina_a', 'vitamina_c', 'sodio'
]

df_demanda_nutricional = (
    df_demanda_nutricional[colunas_finais]
    .sort_values(['CO_ENTIDADE', 'grupo_nutricional', 'num_refeicoes'])
    .reset_index(drop=True)
)

print('\n===================================================')
print('DEMANDA NUTRICIONAL POR ESCOLA')
print('===================================================')
display(df_demanda_nutricional.head(15))

df_demanda_nutricional.to_csv(
    output_folder + 'demanda_nutricional_escolas.csv',
    index=False,
    sep=';',
    decimal=',',
    encoding='utf-8-sig'
)

print(f'\nArquivo "demanda_nutricional_escolas.csv" exportado com '
      f'{len(df_demanda_nutricional)} linhas.')

# ------------------------------------------------------------
# VALIDAÇÃO CRUZADA — alunos contados na demanda nutricional
# vs. matrículas computadas no bloco de repasse.
# Diferenças são ESPERADAS e não indicam erro: a demanda
# nutricional inclui escolas diferenciadas dentro de seus grupos
# por idade/etapa e NÃO inclui AEE contraturno; o bloco de
# repasse faz o oposto (regras de elegibilidade financeira).
# ------------------------------------------------------------

total_alunos_nutricional = (
    df_demanda_nutricional.groupby('CO_ENTIDADE')['quantidade_alunos']
    .sum()
    .rename('total_alunos_demanda_nutricional')
)

comparacao_repasse_nutricional = (
    df_resumo_escolas
    .set_index('CO_ENTIDADE')[['NO_ENTIDADE', 'total_matriculas_repasse']]
    .join(total_alunos_nutricional)
)

comparacao_repasse_nutricional['diferenca'] = (
    comparacao_repasse_nutricional['total_alunos_demanda_nutricional']
    - comparacao_repasse_nutricional['total_matriculas_repasse']
)

print('\n===================================================')
print('VALIDAÇÃO CRUZADA — REPASSE vs. DEMANDA NUTRICIONAL')
print('===================================================')
display(
    comparacao_repasse_nutricional
    .sort_values('diferenca', ascending=False)
    .head(10)
)

print(f"\nTotal de alunos (demanda nutricional): "
      f"{total_alunos_nutricional.sum():,.0f}")
print(f"Total de matrículas (repasse FNDE):     "
      f"{df_resumo_escolas['total_matriculas_repasse'].sum():,.0f}")



DEMANDA NUTRICIONAL POR ESCOLA


,CO_ENTIDADE,NO_ENTIDADE,grupo_nutricional,num_refeicoes,quantidade_alunos,energia_kcal,proteina_min,proteina_max,lipidio_min,lipidio_max,carbo_min,carbo_max,calcio,ferro,vitamina_a,vitamina_c,sodio
0,29179319,ESCOLA MUNICIPAL CAMPINAS DE PIRAJA,EJA,1,130.0,477,12,18,8,16,66,77,NaN,NaN,NaN,NaN,600.0
1,29179319,ESCOLA MUNICIPAL CAMPINAS DE PIRAJA,FUND_11_15,1,568.0,473,12,18,13,18,65,77,NaN,NaN,NaN,NaN,600.0
2,29179319,ESCOLA MUNICIPAL CAMPINAS DE PIRAJA,FUND_11_15,3,1.0,1656,41,62,46,64,228,269,NaN,NaN,NaN,NaN,1400.0
3,29179475,CENTRO MUNICIPAL DE EDUCACAO INFANTIL CASA DA ...,CRECHE,3,73.0,708,18,27,20,28,97,115,350.0,2.0,147.0,9.0,NaN
4,29179475,CENTRO MUNICIPAL DE EDUCACAO INFANTIL CASA DA ...,PRE_ESCOLA,1,59.0,270,7,10,8,11,37,44,NaN,NaN,NaN,NaN,600.0
5,29179475,CENTRO MUNICIPAL DE EDUCACAO INFANTIL CASA DA ...,PRE_ESCOLA,3,1.0,945,24,35,26,37,130,154,NaN,NaN,NaN,NaN,1400.0
6,29179670,ESCOLA MUNICIPAL CENTRO COMUNITARIO FREI LEONI...,FUND_6_10,1,131.0,329,8,12,9,13,45,53,NaN,NaN,NaN,NaN,600.0
7,29179670,ESCOLA MUNICIPAL CENTRO COMUNITARIO FREI LEONI...,PRE_ESCOLA,3,15.0,945,24,35,26,37,130,154,NaN,NaN,NaN,NaN,1400.0
8,29180058,CENTRO MUNICIPAL DE EDUCACAO INFANTIL JOAQUIM ...,CRECHE,3,29.0,708,18,27,20,28,97,115,350.0,2.0,147.0,9.0,NaN
9,29180058,CENTRO MUNICIPAL DE EDUCACAO INFANTIL JOAQUIM ...,PRE_ESCOLA,3,79.0,945,24,35,26,37,130,154,NaN,NaN,NaN,NaN,1400.0



Arquivo "demanda_nutricional_escolas.csv" exportado com 1021 linhas.

VALIDAÇÃO CRUZADA — REPASSE vs. DEMANDA NUTRICIONAL


,NO_ENTIDADE,total_matriculas_repasse,total_alunos_demanda_nutricional,diferenca
CO_ENTIDADE,,,,
29179319,ESCOLA MUNICIPAL CAMPINAS DE PIRAJA,699.0,699.0,0.0
29179475,CENTRO MUNICIPAL DE EDUCACAO INFANTIL CASA DA ...,133.0,133.0,0.0
29179670,ESCOLA MUNICIPAL CENTRO COMUNITARIO FREI LEONI...,146.0,146.0,0.0
29180058,CENTRO MUNICIPAL DE EDUCACAO INFANTIL JOAQUIM ...,108.0,108.0,0.0
29180066,ESCOLA MUNICIPAL ELOYNA BARRADAS,429.0,429.0,0.0
29180244,ESCOLA MUNICIPAL PROFESSORA SUZANA IMBASSAHY,281.0,281.0,0.0
29181224,ESCOLA MUNICIPAL CENTRO SOCIAL NEUSA NERY,147.0,147.0,0.0
29181232,ESCOLA MUNICIPAL SANTA TEREZINHA,180.0,180.0,0.0
29181259,CENTRO MUNICIPAL DE EDUCACAO INFANTIL JOSE REN...,211.0,211.0,0.0



Total de alunos (demanda nutricional): 133,523
Total de matrículas (repasse FNDE):     133,523


<a id="resumo-municipal"></a>
## Resumo municipal e validação cruzada


In [9]:
# Inspeção rápida de tipos e nulos do resumo por escola gerado na seção anterior.
df_resumo_escolas.info()

<class 'pandas.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CO_ENTIDADE                414 non-null    int64  
 1   NO_ENTIDADE                414 non-null    str    
 2   matricula_fund_med_eja     414 non-null    float64
 3   matricula_infpre_comum     414 non-null    float64
 4   matricula_diferenciada     414 non-null    float64
 5   matricula_integral         414 non-null    float64
 6   matricula_creche           414 non-null    float64
 7   matricula_complementacao   414 non-null    float64
 8   matricula_aee_contraturno  414 non-null    float64
 9   ref_dia_fund_med_eja       414 non-null    float64
 10  ref_dia_infpre_comum       414 non-null    float64
 11  ref_dia_diferenciada       414 non-null    float64
 12  ref_dia_integral           414 non-null    float64
 13  ref_dia_creche             414 non-null    float64
 14  ref_d

In [10]:
# ------------------------------------------------------------
# Agregação final: totais no nível do município (soma de todas as
# escolas da amostra). Não gera CSV — é apenas uma visualização de
# fechamento para conferência rápida dos números do estudo de caso.
# ------------------------------------------------------------
# ============================================================
# RESUMO DA DEMANDA MUNICIPAL
# ============================================================

resumo_municipio = pd.DataFrame([{
    'matriculas_total': df_censo_amostra['total_matriculas_repasse'].sum(),
    'refeicoes_dia': df_censo_amostra['total_refeicoes_dia'].sum(),
    'refeicoes_ano': df_censo_amostra['total_refeicoes_ano'].sum(),
    'repasse_anual': df_censo_amostra['valor_total_repasse_anual'].sum()
}])

display(resumo_municipio)

,matriculas_total,refeicoes_dia,refeicoes_ano,repasse_anual
0,133523.0,165145.0,33029000.0,19774544.0


<a id="observacoes"></a>
## Observações, limitações conhecidas e pontos em aberto para o TCC

Consolidação, em um único lugar, dos pontos sinalizados ao longo do notebook (`⚠️`/`TODO`), para facilitar a revisão com a orientação antes da entrega final. **Nenhuma dessas correções foi aplicada nesta revisão**, para não alterar o conteúdo dos CSVs de entrada ou de saída sem validação prévia.

### 1. Regras de elegibilidade ainda não confirmadas (`VERIFICAR`)
Dois cálculos no bloco de repasse financeiro têm regra de elegibilidade pendente de confirmação:
- `matricula_complementacao` (usa `QT_MAT_MED_INT` como proxy do Programa de Fomento às Escolas de Ensino Médio em Tempo Integral — pode superestimar, pois nem todo aluno de médio integral participa do programa).
- `matricula_aee_contraturno` (usa `QT_MAT_ESP_CE` — verificar se é de fato equivalente a "AEE em contraturno" ou se mistura outras modalidades de atendimento especializado).

### 2. Premissas já validadas (mantidas como estão)
Registradas na seção [Demanda nutricional](#demanda-nutricional) e reproduzidas aqui por completude:
- Creche: 100% das matrículas usam a referência de Creche (1–3 anos).
- EJA: referência fixada em EJA (19 a 30 anos).
- AEE em contraturno e localização diferenciada não geram grupo nutricional próprio nesta entrega.

### 3. Validação cruzada — diferença esperada
A diferença entre `total_matriculas_repasse` e `total_alunos_demanda_nutricional` (ver final da seção [Demanda nutricional](#demanda-nutricional)) é **esperada e não é um erro**: os dois blocos aplicam regras de contagem distintas (elegibilidade financeira vs. necessidade nutricional real). Vale incluir essa explicação também no corpo do TCC, para antecipar a dúvida da banca.

### Sugestão de próximos passos até a entrega
1. Resolver os itens 1–3 acima com a orientação.
2. Rodar a seção [Dados faltantes](#dados-faltantes) com a base completa e decidir a estratégia de tratamento (hoje é só diagnóstico).
3. Unir este notebook, na narrativa do TCC, com o notebook de dados de alimentos e composição nutricional, deixando explícito no texto que ambos alimentam as mesmas restrições do modelo de Programação Linear (demanda aqui; oferta/custo no notebook de alimentos).
